In [ ]:
!pip install langchain
!pip install chromadb
!pip install sentence-transformers
!pip install transformers
!pip install streamlit
!pip install fastapi
!pip install pypdf

In [ ]:
import pkg_resources

installed = sorted(
    [pkg.project_name for pkg in pkg_resources.working_set]
)

print(installed)

In [ ]:
import os

folders = [
    "data",
    "vector_store"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created")

In [ ]:
os.listdir()

In [ ]:
import os

print(os.listdir("data"))

In [ ]:
!pip install langchain-community

In [ ]:
%pip install pypdf

In [ ]:
import os

print(os.getcwd())

In [ ]:
import os

print(os.listdir())

In [ ]:
print(os.listdir("data"))

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    "data/PDF 1.pdf"
)

docs = loader.load()

print("Loaded pages:", len(docs))

In [ ]:
print(docs[0].page_content[:1000])

In [ ]:
%pip install langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print("Chunks:", len(chunks))

In [ ]:
print(chunks[0].page_content[:500])

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings ready")

In [ ]:
%pip install faiss-cpu

In [ ]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(
    chunks,
    embedding
)

print("Vector database created")

In [ ]:
db.save_local("vector_store")

print("Saved")

In [ ]:
import os

print(os.listdir("vector_store"))

In [ ]:
from langchain_community.vectorstores import FAISS

db = FAISS.load_local(
    "vector_store",
    embedding,
    allow_dangerous_deserialization=True
)

print("Vector store loaded")

In [ ]:
retriever = db.as_retriever()

print("Retriever ready")

In [ ]:
query = "How should people respond during floods?"

docs_found = retriever.invoke(query)

print(docs_found[0].page_content[:1000])

In [ ]:
%pip install openai langchain-openai python-dotenv

In [ ]:
%pip install transformers torch accelerate

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

print("Local model ready")

In [ ]:
retriever = db.as_retriever()

print("Retriever ready")

In [ ]:
def ask_disaster_bot(question):

    docs = retriever.invoke(question)

    context = "\n".join(
        [doc.page_content for doc in docs[:3]]
    )

    prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

    output = generator(
        prompt,
        max_new_tokens=120,
        truncation=True
    )

    return output[0]["generated_text"]

In [ ]:
answer = ask_disaster_bot(
    "What should people do during floods?"
)

print(answer)

In [ ]:
print(
    ask_disaster_bot(
        "How should people prepare before an earthquake?"
    )
)